In [ ]:
import os, json, re
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path("/Users/nick/Desktop/intern/POLYMARKET")
os.chdir(BASE)

con = duckdb.connect()
con.execute("SET threads=4; SET memory_limit='2GB';")

# ── Lookup tables ──────────────────────────────────────────────────────────────

# 1. condition_id → slug / market_name / bucket  (full historical set)
clob = json.loads((BASE / "all_elon_cids_2026.json").read_text())
cid_to_slug = {m["condition_id"]: m["market_slug"] for m in clob["markets"]
               if m.get("condition_id") and m.get("market_slug")}

def slug_to_bucket(slug):
    m = re.search(r"(\d+-\d+)$", slug)
    if m: return m.group(1)
    m = re.search(r"(\d+plus)$", slug)
    if m: return m.group(1)
    return None

def slug_to_market(slug):
    m = re.search(r"of-tweets-(january|february|march|april|may|june|july|august|"
                  r"september|october|november|december)-(\d{4})-", slug)
    if m: return f"{m.group(1).capitalize()} {m.group(2)} monthly"
    m = re.search(r"of-tweets-([a-z]+-\d+)-([a-z]+-\d+)", slug)
    if m: return f"{m.group(1)} – {m.group(2)} weekly"
    return slug

cid_to_bucket = {cid: slug_to_bucket(s) for cid, s in cid_to_slug.items()}
cid_to_market = {cid: slug_to_market(s) for cid, s in cid_to_slug.items()}

# 2. asset_id (token) → YES/NO  (from market_ids.json — active markets only,
#    but enough to label the tokens we care about)
mids = json.loads((BASE / "market_ids.json").read_text())
token_to_side = {}
for m in mids:
    if m.get("yes_token"): token_to_side[m["yes_token"]] = "YES"
    if m.get("no_token"):  token_to_side[m["no_token"]]  = "NO"

print(f"condition_id → slug mappings : {len(cid_to_slug):,}")
print(f"token → YES/NO mappings      : {len(token_to_side):,}")
print(f"staging files                : {len(list(BASE.glob('staging/src_*.parquet')))}")

In [ ]:
# ── Load one day ──────────────────────────────────────────────────────────────
DAY = "2026-05-10"

day = con.execute(f"""
    SELECT *
    FROM '{BASE}/elon_tweet_ticks.parquet'
    WHERE timestamp >= '{DAY}'
      AND timestamp <  '{DAY}'::DATE + INTERVAL 1 DAY
""").df()

# Attach human labels
day["bucket"]      = day["market"].map(cid_to_bucket)
day["market_name"] = day["market"].map(cid_to_market)
day["token_side"]  = day["asset_id"].map(token_to_side)   # YES / NO / NaN

print(f"{len(day):,} rows  |  {day.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nEvent types:")
print(day["event_type"].value_counts().to_string())
print(f"\nToken sides captured:")
print(day["token_side"].value_counts(dropna=False).to_string())
print(f"\nBuckets with most activity:")
print(day[day["bucket"].notna()].groupby("bucket").size().sort_values(ascending=False).head(8).to_string())

In [ ]:
# ── Trade summary for the day ─────────────────────────────────────────────────
# Both YES and NO tokens are captured (2 tokens per condition_id).
# best_bid/best_ask on the YES token directly give the implied probability.
# NO token best_bid ≈ 1 - YES best_ask (complementary).

trades = day[day["event_type"] == "last_trade_price"].copy()

summary = (
    trades
    .groupby(["bucket", "token_side"])
    .agg(
        n_trades  = ("price", "count"),
        volume    = ("size", "sum"),
        avg_price = ("price", "mean"),
        min_price = ("price", "min"),
        max_price = ("price", "max"),
    )
    .reset_index()
    .sort_values(["bucket", "token_side"])
)

print(f"Total trades on {DAY}: {len(trades)}  (YES + NO tokens combined)")
print(f"Total volume: ${trades['size'].sum():,.0f} USDC\n")
print(summary.to_string(index=False))

In [ ]:
# ── Mid-price trajectory: YES token only, 5-min resampled ────────────────────
# Use YES token for mid-price (best_bid = prob you can sell YES, best_ask = prob to buy YES)
# Filter to price_change events with a valid book (best_bid is populated)

BUCKET = "840-879"

yes_book = day[
    (day["event_type"] == "price_change") &
    (day["bucket"] == BUCKET) &
    (day["token_side"] == "YES") &
    day["best_bid"].notna()
].copy()

yes_book["mid"]    = (yes_book["best_bid"] + yes_book["best_ask"]) / 2
yes_book["spread"] =  yes_book["best_ask"] - yes_book["best_bid"]

print(f"{BUCKET} YES token on {DAY}: {len(yes_book):,} orderbook updates")
print(f"  mid range: {yes_book['mid'].min():.4f} – {yes_book['mid'].max():.4f}")
print(f"  avg spread: {yes_book['spread'].mean():.4f}")
yes_book[["timestamp", "mid", "spread", "best_bid", "best_ask"]].head(8)

In [ ]:
# ── Plot 1: Mid-price across buckets (one day) ───────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

# Panel A: bucket probability distribution (5-min resampled mid)
buckets_to_plot = ["800-839", "840-879", "880-919", "920-959", "960-999"]
for bucket in buckets_to_plot:
    sub = day[
        (day["event_type"] == "price_change") &
        (day["bucket"] == bucket) &
        (day["token_side"] == "YES") &
        day["best_bid"].notna()
    ].copy()
    if sub.empty: continue
    sub["mid"] = (sub["best_bid"] + sub["best_ask"]) / 2
    ts = sub.set_index("timestamp")["mid"].resample("5min").last().ffill()
    axes[0].plot(ts.index, ts.values, label=bucket, linewidth=1.2)
axes[0].set_title(f"YES token mid-price by bucket — {DAY}")
axes[0].set_ylabel("Implied probability")
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Panel B: actual trades (YES + NO) for leading bucket
bucket_trades = trades[trades["bucket"] == BUCKET].copy()
colors = {"YES": "steelblue", "NO": "salmon"}
for side, grp in bucket_trades.groupby("token_side"):
    axes[1].scatter(grp["timestamp"], grp["price"],
                    s=grp["size"].clip(1, 300), alpha=0.7,
                    color=colors.get(side, "gray"), label=f"{side} trades")
axes[1].set_title(f"Trades — {BUCKET} bucket  (dot area = USDC notional)")
axes[1].set_ylabel("Fill price"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Panel C: daily trade count & volume across full 25-day dataset
daily = con.execute(f"""
    SELECT
        DATE_TRUNC('day', timestamp)::DATE AS day,
        COUNT(*) FILTER (WHERE event_type='last_trade_price') AS trades,
        SUM(CAST(size AS DOUBLE)) FILTER (WHERE event_type='last_trade_price') AS volume
    FROM '{BASE}/elon_tweet_ticks.parquet'
    GROUP BY 1 ORDER BY 1
""").df()
ax3b = axes[2].twinx()
axes[2].bar(daily["day"].astype(str), daily["trades"], color="steelblue", alpha=0.6, label="trades")
ax3b.plot(daily["day"].astype(str), daily["volume"] / 1000, color="orange", linewidth=1.5, label="volume ($k)")
axes[2].set_title("Daily trades & volume — full dataset (May 1–25)")
axes[2].set_ylabel("Trades"); ax3b.set_ylabel("Volume ($k USDC)")
axes[2].tick_params(axis="x", rotation=45)
axes[2].legend(loc="upper left", fontsize=8); ax3b.legend(loc="upper right", fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()